# Import

In [ ]:
import pandas as pd
import numpy as np
import pickle
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from tqdm import tqdm
from PIL import Image
from sklearn.model_selection import train_test_split
import sys, os
sys.path.append(os.path.abspath(".."))

from src.preprocessing import load_dataset, build_gene_matrix
from src.gradcam_utils import (
    make_grad_heatmap, display_gradcam_with_img,
    compute_group_heatmap, compute_difference_heatmap, resize_and_normalize_heatmap,
    plot_heatmap_with_max_region, plot_heatmap_diff,
    plot_diff_heatmap_with_top_genes,
    get_gene_pathway_from_position, get_top_n_bright_spots, find_gene_position,
    get_top_positive_diff_spots, get_top_negative_diff_spots,
    plot_diff_heatmap_subset
)

# Load data

In [ ]:
# Numpy 배열 불러오기 
X_responder = np.load("../data/npy/X_responder.npy")
y_responder = np.load("../data/npy/y_responder.npy")
X_nonresponder = np.load("../data/npy/X_nonresponder.npy")
y_nonresponder = np.load("../data/npy/y_nonresponder.npy")

print("Responder:", X_responder.shape, "Non-responder:", X_nonresponder.shape)

# gene x pathway 
dataset_path = "../data/dataset.csv"
X, y = load_dataset(dataset_path)
with open("../data/pathway_list.pkl", "rb") as f:
    pathway_list = pickle.load(f)   
with open("../data/cell_list.pkl", "rb") as f:
    cell_list = pickle.load(f)   
    
gene_matrix = build_gene_matrix(X, pathway_list=pathway_list, cell_list=cell_list, max_genes=383)

# Load 2D CNN Model

In [ ]:
# 학습된 2D CNN 모델 로드
model_cnn2d = keras.models.load_model("../models/cnn2d_best.h5")

In [ ]:
model_cnn2d.summary()

In [ ]:
last_conv_layer_name = "max_pooling2d_2"  

# Grad-CAM

## Responder heatmap

In [ ]:
res_heatmap = compute_group_heatmap(X_responder, model_cnn2d, last_conv_layer_name)
res_heatmap = resize_and_normalize_heatmap(res_heatmap, target_shape=(383,186))
plt.imshow(res_heatmap, cmap="viridis"); plt.title("Responder Avg Heatmap"); plt.show()

In [ ]:
top_spots_res = plot_heatmap_with_max_region(
    res_heatmap,
    gene_matrix,
    pathway_list,
    top_n=8
)

## Non-responder heatmap

In [ ]:
non_heatmap = compute_group_heatmap(X_nonresponder, model_cnn2d, last_conv_layer_name)
non_heatmap = resize_and_normalize_heatmap(non_heatmap, target_shape=(383,186))
plt.imshow(non_heatmap, cmap="viridis"); plt.title("Non-responder Avg Heatmap"); plt.show()

In [ ]:
top_spots_nonres = plot_heatmap_with_max_region(
    non_heatmap,
    gene_matrix,
    pathway_list,
    top_n=8
)

## Difference

In [ ]:
heatmap_diff = plot_heatmap_diff(res_heatmap, non_heatmap)

In [ ]:
top_diff = plot_diff_heatmap_with_top_genes(
    heatmap_diff,
    gene_matrix,
    pathway_list,
    top_n=9
)

## Responder > Non-responder

In [ ]:
top_pos = get_top_positive_diff_spots(heatmap_diff, gene_matrix, pathway_list, top_n=15)
plot_diff_heatmap_subset(
    heatmap_diff,
    gene_matrix,
    pathway_list,
    top_pos,
    title="Responder > Non-responder"
)

## Non-responder > Responder

In [ ]:
# Non-responder 강조 (음수 차이)
top_neg = get_top_negative_diff_spots(heatmap_diff, gene_matrix, pathway_list, top_n=9)
plot_diff_heatmap_subset(
    heatmap_diff,
    gene_matrix,
    pathway_list,
    top_neg,
    title="Non-responder > Responder"
)